<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/02_construction_data_preparation_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การเตรียมข้อมูลรายการจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 จากไฟล์ต้นทาง 8 ไฟล์

Notebook นี้รวมไฟล์ สกัดรายการจ้างก่อสร้าง และสร้างข้อมูลสองระดับ:

- ระดับโครงการ–ผู้รับจ้าง–สัญญา
- ระดับโครงการ


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
base_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/'
    'egp-contract/2569'
)

processed_dir = base_dir.parent / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(base_dir.glob('*.csv'))

contract_supplier_path = (
    processed_dir
    / 'construction_contract_supplier_records_2569.csv'
)

project_summary_path = (
    processed_dir
    / 'construction_project_summary_2569.csv'
)

project_dir = base_dir.parents[3]
figure_dir = project_dir / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

print(f'CSV files: {len(csv_files)}')
print(f'Contract-supplier output: {contract_supplier_path}')
print(f'Project summary output: {project_summary_path}')


## 1. ตรวจโครงสร้างไฟล์ต้นทาง

อ่านตัวอย่างจากไฟล์แรกเพื่อตรวจชื่อคอลัมน์และรูปแบบข้อมูล


In [ ]:
sample_data = pd.read_csv(csv_files[0], nrows=5)

print(f'Sample file: {csv_files[0].name}')
print(f'Columns: {sample_data.shape[1]}')

display(sample_data)

In [ ]:
for number, column in enumerate(sample_data.columns, start=1):
    print(f'{number:02d}. {column}')

## 2. รวมไฟล์และสกัดรายการจ้างก่อสร้าง

อ่านข้อมูลทีละไฟล์ นับประเภทโครงการ และเก็บเฉพาะรายการจ้างก่อสร้าง


In [ ]:
project_type_column = 'ชื่อประเภทโครงการ'

type_counts_list = []
construction_parts = []
file_summaries = []

for file_path in csv_files:
    data = pd.read_csv(file_path, low_memory=False)
    data.columns = data.columns.str.strip()

    project_type = data[project_type_column].astype('string')
    project_type = project_type.str.strip().fillna('ไม่ระบุ')

    type_counts_list.append(project_type.value_counts())

    construction = data[project_type == 'จ้างก่อสร้าง'].copy()
    construction['source_file'] = file_path.name
    construction_parts.append(construction)

    file_summaries.append({
        'file_name': file_path.name,
        'total_rows': len(data),
        'construction_rows': len(construction)
    })

    print(f'{file_path.name}: {len(data):,} rows | 'f'{len(construction):,} construction rows')

In [ ]:
processing_summary = pd.DataFrame(file_summaries)
total_records = processing_summary['total_rows'].sum()

construction_data = pd.concat(construction_parts,ignore_index=True)

total_construction_records = len(construction_data)
construction_pct = total_construction_records / total_records * 100

print(f'Total records: {total_records:,}')
print(f'Construction records: {total_construction_records:,}')
print(f'Construction share: {construction_pct:.2f}%')

### จำนวนรายการตามประเภทโครงการ

เปรียบเทียบสัดส่วนจำนวนรายการของแต่ละประเภทกับข้อมูลทั้งหมด


In [ ]:
type_counts = pd.concat(type_counts_list, axis=1).fillna(0)
type_counts = type_counts.sum(axis=1).astype('int64')
type_counts = type_counts.sort_values(ascending=False)

project_type_counts = type_counts.reset_index()
project_type_counts.columns = [project_type_column, 'record_count']

project_type_counts['record_pct'] = (project_type_counts['record_count'] / total_records * 100)
display(project_type_counts)

In [ ]:
# ดาวน์โหลดฟอนต์สำหรับแสดงภาษาไทยในกราฟ
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

BLUE = '#5B7FA3'
ORANGE = '#D9822B'
GRAY = '#B8C2CC'
TEXT = '#344054'
MUTED = '#667085'
GRID = '#E4E7EC'


In [ ]:
plot_data = (
    project_type_counts
    .head(4)
    .sort_values('record_pct')
    .reset_index(drop=True)
)

colors = [
    ORANGE if project_type == 'จ้างก่อสร้าง' else GRAY
    for project_type in plot_data[project_type_column]
]


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

bars = ax.barh(
    plot_data[project_type_column],
    plot_data['record_pct'],
    color=colors,
    height=0.58
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f} ({percentage:.2f}%)'
        for count, percentage in zip(
            plot_data['record_count'],
            plot_data['record_pct']
        )
    ],
    padding=5,
    fontsize=11,
    color=TEXT
)

ax.set_xlim(0, plot_data['record_pct'].max() * 1.18)
ax.set_title(
    'จำนวนรายการจัดซื้อจัดจ้างตามประเภทโครงการ',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('สัดส่วนจำนวนรายการ (%)')
ax.set_ylabel('ประเภทโครงการ')

ax.grid(axis='x', color=GRID, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.text(
    0.01,
    0.01,
    f'ฐาน: {total_records:,.0f} รายการ | ข้อมูลสะสมถึง 30 กรกฎาคม 2569',
    fontsize=10,
    color=MUTED
)

fig.tight_layout(rect=[0, 0.04, 1, 1])


In [ ]:
png_path = figure_dir / 'fig02_01_procurement_records_by_type.png'
svg_path = figure_dir / 'fig02_01_procurement_records_by_type.svg'

fig.savefig(
    png_path,
    dpi=180,
    bbox_inches='tight',
    facecolor='white'
)
fig.savefig(
    svg_path,
    bbox_inches='tight',
    facecolor='white'
)

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


### ผลการสกัด

ผลด้านบนเป็นจำนวนแถวจากไฟล์ต้นทาง ยังไม่ใช่จำนวนโครงการหรือจำนวนสัญญาไม่ซ้ำ


## 3. สร้างข้อมูลระดับโครงการ–ผู้รับจ้าง–สัญญา

กำหนดหนึ่งรายการไม่ซ้ำด้วยรหัสโครงการ รหัสผู้รับจ้าง และเลขที่สัญญา


In [ ]:
project_id_column = 'รหัสโครงการ'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
contract_column = 'เลขที่สัญญา'
contract_value_column = 'วงเงินงบประมาณในสัญญา (บาท)'

unique_key = [
    project_id_column,
    supplier_id_column,
    contract_column
]

data_summary = pd.Series({
    'รายการจ้างก่อสร้าง': len(construction_data),
    'โครงการไม่ซ้ำ': (
        construction_data[project_id_column].nunique()
    ),
    'สัญญาไม่ซ้ำ': (
        construction_data[contract_column].nunique()
    ),
    'ผู้รับจ้างไม่ซ้ำ': (
        construction_data[supplier_id_column].nunique()
    )
}, name='ค่า')

display(data_summary.to_frame())


In [ ]:
project_row_counts = (
    construction_data[project_id_column]
    .value_counts()
)

print(
    'โครงการที่มีมากกว่า 1 แถว: '
    f'{project_row_counts.gt(1).sum():,}'
)
print(
    'จำนวนแถวสูงสุดต่อโครงการ: '
    f'{project_row_counts.max():,}'
)


In [ ]:
repeated_project_ids = (
    project_row_counts[project_row_counts.gt(1)]
    .head(3)
    .index
)

columns_to_show = [
    project_id_column,
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    supplier_id_column,
    'ชื่อผู้ชนะการเสนอราคา',
    contract_column,
    contract_value_column
]

repeated_project_sample = construction_data.loc[
    construction_data[project_id_column]
    .isin(repeated_project_ids),
    columns_to_show
]

display(repeated_project_sample)


### ตัดรหัสผู้รับจ้างขึ้นต้นด้วย D และตัดแถวซ้ำ

ดำเนินการตามลำดับดังนี้:

1. ตัดแถวที่รหัสผู้รับจ้างขึ้นต้นด้วย `D`
2. ตัดแถวซ้ำด้วย `รหัสโครงการ + รหัสผู้รับจ้าง + เลขที่สัญญา`
3. เก็บแถวแรกของแต่ละชุด

หลายสัญญาและหลายผู้รับจ้างภายในโครงการเดียวยังคงอยู่ในข้อมูล


In [ ]:
supplier_id = (
    construction_data[supplier_id_column]
    .astype('string')
    .str.strip()
)

joint_venture_mask = supplier_id.str.startswith(
    'D',
    na=False
)

joint_venture_rows = construction_data.loc[
    joint_venture_mask,
    columns_to_show
]

print(
    'แถวรหัสผู้รับจ้างขึ้นต้นด้วย D: '
    f'{len(joint_venture_rows):,}'
)

display(joint_venture_rows.head(20))


In [ ]:
contract_supplier_data = (
    construction_data.loc[~joint_venture_mask]
    .copy()
)

print(
    'รายการก่อนตัดรหัส D: '
    f'{len(construction_data):,}'
)
print(
    'แถวรหัส D ที่ตัดออก: '
    f'{joint_venture_mask.sum():,}'
)
print(
    'รายการหลังตัดรหัส D: '
    f'{len(contract_supplier_data):,}'
)


In [ ]:
rows_before_deduplication = len(
    contract_supplier_data
)

contract_supplier_data = (
    contract_supplier_data
    .drop_duplicates(
        subset=unique_key,
        keep='first'
    )
    .copy()
)

removed_duplicate_rows = (
    rows_before_deduplication
    - len(contract_supplier_data)
)

print(
    'รายการก่อนตัดแถวซ้ำ: '
    f'{rows_before_deduplication:,}'
)
print(
    'แถวซ้ำที่ตัดออก: '
    f'{removed_duplicate_rows:,}'
)
print(
    'รายการที่เหลือ: '
    f'{len(contract_supplier_data):,}'
)


## 4. ทำความสะอาดฟิลด์ที่ใช้วิเคราะห์

ปรับข้อความและชนิดข้อมูลของมูลค่าสัญญา ก่อนสร้างข้อมูลระดับโครงการ


In [ ]:
agency_column = 'ชื่อหน่วยงาน'
subagency_column = 'ชื่อหน่วยงานย่อย'
province_column = 'จังหวัด'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'

text_columns = [
    agency_column,
    subagency_column,
    province_column,
    supplier_id_column,
    supplier_name_column,
    contract_column
]

for column in text_columns:
    contract_supplier_data[column] = (
        contract_supplier_data[column]
        .astype('string')
        .str.strip()
        .replace('', pd.NA)
    )

contract_supplier_data[contract_value_column] = (
    pd.to_numeric(
        contract_supplier_data[contract_value_column],
        errors='coerce'
    )
)


## 5. สร้างข้อมูลระดับโครงการ

รวมวงเงินงบประมาณในสัญญาของแต่ละโครงการ และเก็บคุณลักษณะโครงการหนึ่งแถวต่อรหัสโครงการ


In [ ]:
project_columns = [
    project_id_column,
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    'ชื่อประเภทโครงการ',
    agency_column,
    subagency_column,
    'ชื่อวิธีการจัดซื้อจัดจ้าง',
    'ชื่อกลุ่มวิธีการจัดซื้อจัดจ้าง',
    'วันที่ประกาศจัดซื้อจัดจ้าง',
    'ราคากลาง (บาท)',
    'ปีงบประมาณ',
    'วันที่เกิดรายการ',
    province_column,
    'เขต/อำเภอ',
    'แขวง/ตำบล',
    'สถานะโครงการ',
    'พิกัดของโครงการ',
    'ละติจูดของโครงการ',
    'ลองจิจูดของโครงการ'
]

project_attributes = (
    contract_supplier_data[project_columns]
    .drop_duplicates(
        subset=project_id_column,
        keep='first'
    )
)

project_contract_values = (
    contract_supplier_data
    .groupby(project_id_column)[contract_value_column]
    .sum()
    .reset_index(
        name='วงเงินสัญญารวมต่อโครงการ (บาท)'
    )
)

project_summary_data = project_attributes.merge(
    project_contract_values,
    on=project_id_column,
    how='left'
)

print(
    'Contract-supplier shape: '
    f'{contract_supplier_data.shape}'
)
print(
    'Project summary shape: '
    f'{project_summary_data.shape}'
)

display(contract_supplier_data.head())
display(project_summary_data.head())


In [ ]:
contract_supplier_data.to_csv(
    contract_supplier_path,
    index=False,
    encoding='utf-8-sig'
)

project_summary_data.to_csv(
    project_summary_path,
    index=False,
    encoding='utf-8-sig'
)

print(f'Saved: {contract_supplier_path}')
print(f'Saved: {project_summary_path}')


## ไฟล์ผลลัพธ์

| ไฟล์ | ระดับข้อมูล |
|---|---|
| `construction_contract_supplier_records_2569.csv` | หนึ่งแถวต่อโครงการ–ผู้รับจ้าง–สัญญา |
| `construction_project_summary_2569.csv` | หนึ่งแถวต่อโครงการ พร้อมวงเงินสัญญารวมต่อโครงการ |

Notebook 03 ใช้ไฟล์ระดับโครงการสำหรับ EDA ส่วน Notebook 04 ใช้ไฟล์ทั้งสองระดับสำหรับตรวจ Pattern
